In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

import os
from prettytable import PrettyTable
from IPython.display import display

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import DBSCAN, KMeans

from sklearn.decomposition import PCA
import umap.umap_ as umap
import pacmap

base = os.getcwd().replace('\\','/')

# Preprocessing

In [5]:
def sumarryDataFromFile(data):
    mofTable                = PrettyTable()
    mofTable.field_names    = ["No","MOF", "startingTime", "endingTime", "staringRow","endingRow"]

    no                      = 1
    usefulMOF               = []
    mofCol                  = data['Modo de Falla']
    preMOF                  = None
    beginingRow             = 0
    beginingTime            = 0


    for idx in range(len(data)):
        curIdx = idx
        preIdx = idx -1

        if (mofCol[curIdx] != preMOF) and curIdx!=0:
            # --------------------------
            # Collect the last state
            # --------------------------
            endingTime      = data['tiempo'][preIdx] 
            endingRow       = preIdx
            
            if not pd.isna(preMOF):
                mofTable.add_row([no,preMOF, beginingTime,endingTime,beginingRow+2,endingRow+2])
                no +=1
                
                if preMOF not in usefulMOF:
                    usefulMOF.append(preMOF)
            
            # --------------------------
            # Assign new state
            # --------------------------
            preMOF          = mofCol[curIdx]
            beginingTime    = data['tiempo'][curIdx]
            beginingRow     = curIdx
            
    
    print("All the mode of failure:",usefulMOF)
    print(mofTable)

In [11]:
input_variables = ['corriente(A)', 'velocidad(rpm)', 'Potencia(kW)', 'posicion_poste(%)', 'presion_estanque_aire(bar)', 'Eje_piñón_aceite(%)',
                   'Lubricación_principal(psi)', 'Lubricación_principal(A)', 'Lubricación_respaldo(psi)', 'Lubricación_respaldo(A)',
                   'Lubricación_post_filtros(psi)', 'Lubricación_válvula_MPS(psi)',  'Lubricación_entrada_sup(psi)',
                    'Lubricación_entrada_inf(psi)', 'Lubricación_entrada_sup(m3/h)', 'Lubricación_entrada_inf(m3/h)', 'Lubricación_depósito(%)',
                    'Motor_Fase1(A)', 'Motor_Fase2(A)', 'Motor_Fase3(A)', 'Motor_Fase1(V)', 'Motor_Fase2(V)', 'Motor_Fase3(V)']

usedCriticalMOF = [ "Válvula defectuosa",
                    "Mantención programada", 
                    "Falta de aceite hidráulico",
                    "Lubricación  deficiente",
                    "Temperatura fuera de rango",
                    "Bomba hidráulica con fuga",
                    "Fuga de aceite",
                    "Falla en sistema hidráulico",
                    "Falla sistema de hidroset-poste",
                    "Mandibulas o revestimiento suelto"
                                   ]

In [6]:
notModeOfFailure  = [  "Stand by",
                                    "Stock de producto lleno",
                                    "Cambio de Turno",
                                    "Tronadura",
                                    "Falla o problemas aguas arriba",
                                    "Falta de equipo carguío mina",
                                    "Otra falla mecánica",
                                    "Limpieza equipo",
                                    "Detención por Enclavamiento",
                                    "Colación",
                                    "Falla o problemas aguas abajo",
                                    "Ajuste de Setting (Asociado al Plan)",
                                    "Falta de inventario",
                                    "Otra falla instrumentación",
                                    "Falta de Equipos de Apoyo",
                                    "Puesta en Marcha o sistema en vació",
                                    "Bloqueo de Equipos",
                                    "Factor Climático",
                                    "Inspección de equipo ante evento operacional (chatarra, falsa alarma etc)",
                                    "Puesta en marcha de equipos o sistemas en vacío",
                                    "Sensor nivel lectura errónea",
                                    "Corte energía interno",
                                    "Esperas de personal de reparación y/o repuestos y/o equipos de apoyo y/o movimiento de traslado (ins)",
                                    "Esperas de personal de reparación y/o repuestos y/o equipos de apoyo y/o movimiento de traslado (mec)",
                                    "Factor Externo (huelgas, manifestaciones, cortes de caminos, etc)",
                                    "Otra falla operacional",
                                    "Sensor de flujo lectura errónea",
                                    "Sobrecarga, Rebalses y atollos"
                                    ]

In [8]:
df_crusher_1_raw = pd.read_excel('data\criticalMOFCrusher1.xlsx')

In [9]:
df_crusher_1 = df_crusher_1_raw.copy()

In [16]:
df_crusher_1[~df_crusher_1['Modo de Falla'].isin([usedCriticalMOF]+[])]

,tiempo,Descripción Equipo,Modo de Falla,Comentario,Especialidad,Motivo,Inicio Reportado,Fin Reportado,corriente(A),velocidad(rpm),...,Lubricación_entrada_sup(m3/h),Lubricación_entrada_inf(m3/h),Lubricación_depósito(%),Motor_Fase1(A),Motor_Fase2(A),Motor_Fase3(A),Motor_Fase1(V),Motor_Fase2(V),Motor_Fase3(V),Equipo
0,2023-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,59.219971,999.742188,...,154.238174,130.145508,82.337708,57.069424,57.232113,58.813461,4098.093262,4103.093262,4103.093262,NaN
1,2023-01-01 00:01:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,84.427246,999.755005,...,146.453125,130.033981,79.997177,79.959366,80.378281,80.540451,4092.229736,4098.905273,4098.905273,NaN
2,2023-01-01 00:02:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,77.473312,999.750000,...,153.976334,131.851730,79.997879,58.349232,58.767887,60.511925,4096.000000,4106.000000,4106.000000,NaN
3,2023-01-01 00:03:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,80.816895,999.755005,...,148.601105,131.235519,79.998573,72.828659,73.661324,73.244995,4097.465820,4103.918457,4103.918457,NaN
4,2023-01-01 00:04:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,65.109100,999.755005,...,157.718750,131.861130,79.999275,70.685883,71.437683,71.853752,4113.744629,4123.160645,4123.160645,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322101,2023-12-31 23:56:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,53.458450,599.871399,...,93.772392,83.065277,83.252899,53.266342,52.266342,53.266342,4004.000000,4000.532715,4000.532715,NaN
322102,2023-12-31 23:57:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,62.451904,599.871399,...,97.534592,79.874443,84.040138,61.791962,60.791962,62.659969,4004.000000,4000.000000,4000.000000,NaN
322103,2023-12-31 23:58:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,78.401672,599.879578,...,100.747658,79.918999,84.827385,65.324387,64.457558,62.857067,4001.603027,3996.937012,3996.937012,NaN
322104,2023-12-31 23:59:00,NaN,NaN,NaN,NaN,NaN,NaT,NaT,71.656059,599.875427,...,92.796516,78.175087,85.614624,53.619785,52.355797,52.619785,4002.812012,4002.152100,4002.152100,NaN


In [17]:
df_crusher_1.loc[df_crusher_1['Modo de Falla'].isin(usedCriticalMOF), 'event'] = df_crusher_1[df_crusher_1['Modo de Falla'].isin(usedCriticalMOF)].groupby(['Inicio Reportado', 'Fin Reportado']).ngroup().astype('str')

In [7]:
df_first = df_crusher_1[df_crusher_1['Modo de Falla'].isin(['Mantención programada']+usedCriticalMOF)].drop_duplicates(keep='first', subset=['Modo de Falla', 'Inicio Reportado', 'Fin Reportado'])
df_last = df_crusher_1[df_crusher_1['Modo de Falla'].isin(['Mantención programada']+usedCriticalMOF)].drop_duplicates(keep='last', subset=['Modo de Falla', 'Inicio Reportado', 'Fin Reportado'])
df_MOF = pd.concat([df_first, df_last]).sort_values(by='tiempo')
del df_first, df_last
df_MOF[['Modo de Falla', 'Inicio Reportado', 'Fin Reportado']]

,Modo de Falla,Inicio Reportado,Fin Reportado
4774,Válvula defectuosa,2023-01-06 10:51:43,2023-01-06 15:13:56
5035,Válvula defectuosa,2023-01-06 10:51:43,2023-01-06 15:13:56
16996,Mantención programada,2023-01-19 00:54:00,2023-01-19 01:53:09
17055,Mantención programada,2023-01-19 00:54:00,2023-01-19 01:53:09
17146,Mantención programada,2023-01-19 03:23:58,2023-01-19 23:12:31
...,...,...,...
294999,Bomba hidráulica con fuga,2023-12-03 08:00:00,2023-12-03 15:17:42
309116,Mantención programada,2023-12-19 03:55:45,2023-12-19 04:18:32
309138,Mantención programada,2023-12-19 03:55:45,2023-12-19 04:18:32
309179,Mantención programada,2023-12-19 04:58:54,2023-12-20 10:39:01


In [8]:
sumarryDataFromFile(df_crusher_1)

All the mode of failure: ['Válvula defectuosa', 'Sobrecarga, Rebalses y atollos', 'Mantención programada', 'Falta de aceite hidráulico', 'Lubricación \xa0deficiente', 'Temperatura fuera de rango', 'Bomba hidráulica con fuga', 'Fuga de aceite', 'Falla en sistema hidráulico', 'Falla sistema de hidroset-poste', 'Mandibulas o revestimiento suelto']
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| No |                MOF                |     startingTime    |      endingTime     | staringRow | endingRow |
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| 1  |         Válvula defectuosa        | 2023-01-06 10:52:00 | 2023-01-06 15:13:00 |    4776    |    5037   |
| 2  |   Sobrecarga, Rebalses y atollos  | 2023-01-08 08:01:00 | 2023-01-08 08:20:00 |    7234    |    7253   |
| 3  |       Mantención programada       | 2023-01-19 00:54:00 | 2023-01-19 01:53:00 |   1699

In [18]:
mask = df_crusher_1['event'].isnull()
groups = mask.ne(mask.shift()).cumsum().astype('str')
df_crusher_1['trajectory'] = np.where(mask, groups, np.nan)

In [21]:
df_crusher_1[df_crusher_1['Modo de Falla']=='Mantención programada']

,tiempo,Descripción Equipo,Modo de Falla,Comentario,Especialidad,Motivo,Inicio Reportado,Fin Reportado,corriente(A),velocidad(rpm),...,Lubricación_depósito(%),Motor_Fase1(A),Motor_Fase2(A),Motor_Fase3(A),Motor_Fase1(V),Motor_Fase2(V),Motor_Fase3(V),Equipo,event,trajectory
16996,2023-01-19 00:54:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-01-19 00:54:00,2023-01-19 01:53:09,13.029175,996.579956,...,61.051174,48.124409,47.161922,48.124409,4107.725586,4141.288086,4141.288086,3200CR001,1,NaN
16997,2023-01-19 00:55:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-01-19 00:54:00,2023-01-19 01:53:09,0.000000,4.014394,...,58.085197,0.000000,0.000000,0.000000,4155.844238,4156.000000,4156.000000,3200CR001,1,NaN
16998,2023-01-19 00:56:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-01-19 00:54:00,2023-01-19 01:53:09,0.000000,4.073257,...,53.321899,0.000000,0.000000,0.000000,4156.000000,4160.811523,4160.811523,3200CR001,1,NaN
16999,2023-01-19 00:57:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-01-19 00:54:00,2023-01-19 01:53:09,0.000000,-2.000000,...,49.536991,0.000000,0.000000,0.000000,4155.844238,4156.194824,4156.194824,3200CR001,1,NaN
17000,2023-01-19 00:58:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-01-19 00:54:00,2023-01-19 01:53:09,0.000000,-2.000000,...,46.782021,0.000000,0.000000,0.000000,4152.150879,4156.188477,4156.188477,3200CR001,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310955,2023-12-20 10:35:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-12-19 04:58:54,2023-12-20 10:39:01,0.000000,-2.000000,...,84.993813,0.000000,0.000000,0.000000,4083.569580,4098.124512,4098.124512,3200CR001,38,NaN
310956,2023-12-20 10:36:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-12-19 04:58:54,2023-12-20 10:39:01,0.000000,-2.000000,...,85.024849,0.000000,0.000000,0.000000,4084.684326,4101.000000,4101.000000,3200CR001,38,NaN
310957,2023-12-20 10:37:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-12-19 04:58:54,2023-12-20 10:39:01,0.000000,-2.000000,...,85.055885,0.000000,0.000000,0.000000,4083.000000,4101.000000,4101.000000,3200CR001,38,NaN
310958,2023-12-20 10:38:00,Chancador Primario 3200CR001,Mantención programada,NaN,Tiempo Mantenimiento Programado,Mantenimiento Programado,2023-12-19 04:58:54,2023-12-20 10:39:01,159.492599,-2.000000,...,85.086929,12.296227,13.049058,12.798115,4085.315674,4101.000000,4101.000000,3200CR001,38,NaN


## Plot period of time series between events

In [19]:
fig = px.line(df_crusher_1, x='tiempo', y='corriente(A)', color='trajectory')
fig.show()

# Preprocessing 2

In [11]:
count=df_crusher_1.groupby('trajectory').size().reset_index(name='count')
df_crusher_1 = pd.merge(df_crusher_1, count, on='trajectory')
df_crusher_1=df_crusher_1[df_crusher_1['count']>1000]
df_crusher_1['count']=df_crusher_1['count'].astype('str')

In [12]:
# Standardize the data
scaler = StandardScaler() # MinMaxScaler() 
scaled_data = scaler.fit_transform(df_crusher_1.select_dtypes('number'))
df_scaled = df_crusher_1.copy()
df_scaled.loc[:, df_crusher_1.select_dtypes('number').columns] = scaled_data
df_input = df_scaled[input_variables]

In [27]:
df_crusher_1['trajectory'].unique()

array(['1', nan, '3', '5', '7', '9', '11', '13', '15', '17', '19', '21',
       '23', '25', '27', '29', '31', '33', '35', '37', '39', '41', '43',
       '45', '47', '49', '51', '53', '55', '57', '59', '61', '63', '65',
       '67', '69', '71'], dtype=object)

In [22]:
a={'com':2, 'eps': 1, 'min':10}

In [24]:
a.com_

AttributeError: 'dict' object has no attribute 'com_'

# Dimensionality reduction and clustering

In [13]:
clusters = {}

for trajectory in df_crusher_1.trajectory.dropna().unique():
    df_input_trajectory = df_input[df_crusher_1['trajectory']==trajectory]

    pca_model = PCA(n_components=3)
    umap_model = umap.UMAP(n_components=3)
    pacmap_model = pacmap.PaCMAP(n_components=3) 

    transformed_data_pca = pca_model.fit_transform(df_input_trajectory)
    transformed_data_umap = umap_model.fit_transform(df_input_trajectory)
    transformed_data_pacmap = pacmap_model.fit_transform(df_input_trajectory)

    dbscan_model = DBSCAN(eps=0.9, min_samples=60, n_jobs=-1)
    clusters_dbscan_pca = dbscan_model.fit_predict(transformed_data_pca)

    dbscan_model = DBSCAN(eps=2.4, min_samples=60, n_jobs=-1)
    clusters_dbscan_umap = dbscan_model.fit_predict(transformed_data_umap)

    dbscan_model = DBSCAN(eps=2.4, min_samples=60, n_jobs=-1)
    clusters_dbscan_pacmap = dbscan_model.fit_predict(transformed_data_pacmap)

    df_input_experiment = df_input_trajectory.copy()
    df_input_experiment['tiempo'] = df_crusher_1[df_crusher_1['trajectory']==trajectory]['tiempo']

    df_input_experiment['pca_x'] = transformed_data_pca[:, 0]
    df_input_experiment['pca_y'] = transformed_data_pca[:, 1]
    df_input_experiment['pca_z'] = transformed_data_pca[:, 2]

    df_input_experiment['umap_x'] = transformed_data_umap[:, 0]
    df_input_experiment['umap_y'] = transformed_data_umap[:, 1]
    df_input_experiment['umap_z'] = transformed_data_umap[:, 2]
    
    df_input_experiment['pacmap_x'] = transformed_data_pacmap[:, 0]
    df_input_experiment['pacmap_y'] = transformed_data_pacmap[:, 1]
    df_input_experiment['pacmap_z'] = transformed_data_pacmap[:, 2]

    df_input_experiment['dbscan_pca'] = clusters_dbscan_pca.astype('str')
    df_input_experiment['dbscan_umap'] = clusters_dbscan_umap.astype('str')
    df_input_experiment['dbscan_pacmap'] = clusters_dbscan_pacmap.astype('str')

    clusters[trajectory]= df_input_experiment
    

c:\Users\CAEX\Desktop\Diego\Multimodal Project\mmodal\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:301: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



In [14]:
cmap = ListedColormap(sns.color_palette("hls", 256).as_hex())
clustering_algorithms=['pca', 'umap', 'pacmap']
trajectories = df_crusher_1.trajectory.unique()

n_rows, n_cols = len(df_crusher_1.trajectory.unique()), 3
for i in range(n_rows):
    fig, axes = plt.subplots(1, n_cols, figsize=(15, 5), subplot_kw={"projection": "3d"})
    trajectory = trajectories[i]
    for j in range(n_cols):
        algorithm = clustering_algorithms[j]

        scatterplot = axes[j].scatter(xs=clusters[trajectory][f'{algorithm}_x'], 
                                      ys=clusters[trajectory][f'{algorithm}_y'], 
                                      zs=clusters[trajectory][f'{algorithm}_z'], 
                                      c=clusters[trajectory][f'dbscan_{algorithm}'].astype('int'),   
                                      s=20, marker='o', cmap=cmap, alpha=0.3)
        
        axes[j].set_xlabel(f'{algorithm}_x')
        axes[j].set_ylabel(f'{algorithm}_y')
        axes[j].set_zlabel(f'{algorithm}_z')
        axes[j].legend(*scatterplot.legend_elements(), loc='upper left')
    fig.suptitle(f'Dimensionality reduction for trajectory {trajectory}', fontsize=10, y=0.88)
        
    plt.savefig(f'clustering_figures\{trajectory}.png')
    plt.close(fig)


In [32]:
import tqdm

In [31]:
trajectory=df_crusher_1.trajectory.unique()[2]
variable=clusters[trajectory].columns[2] #'corriente(A)'
algorithm='pacmap'

fig = px.scatter(clusters[trajectory], x='tiempo', y=variable, color=f'dbscan_{algorithm}')
line_plot = px.line(clusters[trajectory], x='tiempo', y=variable)

fig.add_trace(line_plot.data[0])
fig.update_traces(line=dict(color='black'))
fig.show()

NameError: name 'clusters' is not defined

In [16]:
# algorithm = 'pacmap'
# fig = px.scatter_3d(df_input_experiment, x=f'{algorithm}_x', y=f'{algorithm}_y', 
#                     z=f'{algorithm}_z',
#                     color=f'dbscan_{algorithm}', size_max = 10)
# fig.update_layout(width=600,height=500)
# fig.show()

# Cluster Comparison

In [17]:
from sklearn.metrics.pairwise import euclidean_distances

In [ ]:
def calculateCentroids(data, trajectory, algorithm):
    df_trajectory_reduced = data[trajectory][[f'{algorithm}_x',f'{algorithm}_y', f'{algorithm}_z']]
    labels_trajectory = data[trajectory][f'dbscan_{algorithm}']

In [50]:
trajectories = df_crusher_1.trajectory.unique()
trajectories

algorithm='pacmap'
for trajectory1 in trajectories:
    for trajectory2 in trajectories:
        if trajectory1==trajectory2:
            continue


In [18]:
trajectories = df_crusher_1.trajectory.unique()
i=2
trajectory = trajectories[0]
trajectory2 = trajectories[0+i]
algorithm='pacmap'

In [19]:
df_trajectory_reduced = clusters[trajectory][[f'{algorithm}_x',f'{algorithm}_y', f'{algorithm}_z']]
labels_trayectory = clusters[trajectory][f'dbscan_{algorithm}']

df_trajectory2_reduced = clusters[trajectory2][[f'{algorithm}_x',f'{algorithm}_y', f'{algorithm}_z']]
labels_trayectory2 = clusters[trajectory2][f'dbscan_{algorithm}']

In [20]:
representatives1 = []
for label in np.unique(labels_trayectory):
    if int(label)==-1:
        continue

    cluster_points = df_trajectory_reduced[labels_trayectory == label]
    representative = np.mean(cluster_points, axis=0).values
    representatives1.append(representative)


representatives2 = []
for label in np.unique(labels_trayectory2):
    if int(label)==-1:
        continue

    cluster_points = df_trajectory2_reduced[labels_trayectory2 == label]
    representative = np.mean(cluster_points, axis=0).values
    representatives2.append(representative)

representatives1, representatives2 = np.array(representatives1), np.array(representatives2)

In [21]:
distances = euclidean_distances(representatives1, representatives2)

threshold = 5
similar_clusters = []
for i, row in enumerate(distances):
    for j, distance in enumerate(row):
        if distance < threshold:
            similar_clusters.append((i, j))
            
for cluster_pair in similar_clusters:
    cluster1_index, cluster2_index = cluster_pair
    cluster1_points = df_trajectory_reduced[labels_trayectory == str(cluster1_index)]
    cluster2_points = df_trajectory2_reduced[labels_trayectory2 == str(cluster2_index)]

if len(similar_clusters)!=0:
    cluster1_points.loc[:,'label']=f'cluster: {cluster1_index}, trajectory:{trajectory}'
    cluster2_points.loc[:,'label']=f'cluster: {cluster2_index}, trajectory:{trajectory2}'


C:\Users\CAEX\AppData\Local\Temp\ipykernel_13996\3486302353.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\CAEX\AppData\Local\Temp\ipykernel_13996\3486302353.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
fig = px.scatter_3d(pd.concat([cluster1_points, cluster2_points]), 
                    x=f'{algorithm}_x', 
                    y=f'{algorithm}_y', 
                    z=f'{algorithm}_z',
                    color='label', size_max = 10)
fig.update_layout(width=600,height=500)
fig.show()

In [23]:
df1 = clusters[trajectory][clusters[trajectory][f'dbscan_{algorithm}']==str(cluster1_index)]
df1['label'] = f'cluster: {cluster1_index}, trajectory:{trajectory}'

df2 = clusters[trajectory2][clusters[trajectory2][f'dbscan_{algorithm}']==str(cluster2_index)]
df2['label'] = f'cluster: {cluster2_index}, trajectory:{trajectory2}'

C:\Users\CAEX\AppData\Local\Temp\ipykernel_13996\3796801617.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\CAEX\AppData\Local\Temp\ipykernel_13996\3796801617.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [45]:
trajectory=df_crusher_1.trajectory.unique()[15]
variable=clusters[trajectory].columns[3] #'corriente(A)'
algorithm='pacmap'
df=pd.concat([df1,df2])
fig = px.scatter(df, x='tiempo', y=variable, color='label')
line_plot = px.line(df, x='tiempo', y=variable)

fig.add_trace(line_plot.data[0])
fig.update_traces(line=dict(color='black'))
fig.show()

In [41]:
clusters[trajectory]['dbscan_pacmap'].apply(lambda row: chr(ord('A') + int(row))).replace('@', 'Z')[::10]

176848    A
176858    B
176868    A
176878    A
176888    B
         ..
178178    F
178188    E
178198    B
178208    F
178218    F
Name: dbscan_pacmap, Length: 138, dtype: object

# GPT-2

In [37]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [38]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
# GPT-2
def predict_next_words(sequence, model, tokenizer, num_words):
    # tokenization
    input_ids = tokenizer.encode(sequence, return_tensors='pt')

    # generate next words
    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            max_length=len(input_ids[0]) + num_words, 
            pad_token_id=model.config.eos_token_id,
            #num_return_sequences=1,
            do_sample=True,
            top_k=50,
            top_p=0.95,
        )
    
    # decode words
    next_words = tokenizer.decode(output[0, len(input_ids[0]):], skip_special_tokens=True)
    
    return next_words

In [44]:
sequence = ' '.join(clusters[trajectory]['dbscan_pacmap'].apply(lambda row: chr(ord('A') + int(row))).replace('@', 'Z')[::10]) #'A A A B B A C A A C C B B A A C B A' #"the weather is"
num_words = 10  # number of words to predict

next_words = predict_next_words(sequence, model, tokenizer, num_words)
print(next_words)
print(sequence+next_words)

 F E F F F E F F F F
A B A A B A A B A A B B B B C B C B Z A C B A B B B B B A B C B B B B B A A A A B B B B A A B A A A A A A B B B B A B A B B B B A B B A A A B B B B B A A A B A B B A B B Z B Z B B B B Z B Z Z Z D Z D D Z D Z Z Z D Z D D D Z D Z D D D D Z D Z E F F E E F E E F F F F F E B F F F E F F F E F F F F


### --------

In [25]:
# cmap = ListedColormap(sns.color_palette("hls", 256).as_hex())
# fig, ax = plt.subplots(1, 1, figsize=(5, 5), subplot_kw={"projection": "3d"})

# scatterplot = ax.scatter(xs=representatives1[:,0], 
#                               ys=representatives1[:,1],
#                               zs=representatives1[:,0], 
#                               c=labels_trayectory.astype('int').unique(),   
#                               s=20, marker='o', cmap=cmap)

# scatterplot2 = ax.scatter(xs=representatives2[:,0], 
#                               ys=representatives2[:,1],
#                               zs=representatives2[:,0], 
#                               c=labels_trayectory2.astype('int').unique(),   
#                               s=20, marker='o', cmap=cmap)


# ax.set_xlabel(f'{algorithm}_x')
# ax.set_ylabel(f'{algorithm}_y')
# ax.set_zlabel(f'{algorithm}_z')
# ax.legend(*scatterplot.legend_elements(), loc='upper left')
# ax.legend(*scatterplot2.legend_elements(), loc='upper right')
# ax.set_title(f'Dimensionality reduction for trajectory {trajectory}', fontsize=10, y=0.88)
# plt.show()
